# 01 - Training Data EDA: Synthetic Noise on Real fNIRS

This notebook explores the training data generation pipeline. The core strategy is to take clean, real fNIRS recordings, preprocess them, and then inject various types of synthetic noise on-the-fly during training. This allows for a large and diverse training set.

**Phase 1 Objective:** Build paired (noisy, clean) fNIRS windows for supervised training using real fNIRS recordings with synthetically injected noise.

In [1]:
# Standard library imports
from pathlib import Path
import logging

# Third-party imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Project-specific imports
from src.data.noise_synthesis import (
    synthesize_mayer_wave,
    synthesize_respiratory,
    synthesize_cardiac,
    synthesize_motion_spikes,
    synthesize_electronic,
    inject_noise
)
from src.data.dataset_train import FNIRSDenoiseDataset

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Configure plotting style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

ModuleNotFoundError: No module named 'src'

## 1. Configuration and Paths

Define the directory where the preprocessed clean fNIRS `.npz` files are stored. These files are the output of `src/data/preprocess_train.py`.

In [ ]:
# Path to the directory containing preprocessed training .npz files
PROCESSED_TRAIN_DIR = Path("../data/processed_train")

print(f"Processed train data directory: {PROCESSED_TRAIN_DIR}")

## 2. Load and Visualize a Clean Signal Window

First, let's load one of the preprocessed files and visualize a 'clean' signal window. This is the target that our model will learn to recover.

In [ ]:
npz_files = list(PROCESSED_TRAIN_DIR.glob("*.npz"))

if not npz_files:
    logging.error(f"No .npz files found in {PROCESSED_TRAIN_DIR}. Please run src/data/preprocess_train.py first.")
    raise FileNotFoundError(f"No .npz files found in {PROCESSED_TRAIN_DIR}")

sample_npz_path = npz_files[0]
print(f"Loading sample data from: {sample_npz_path}")

data = np.load(sample_npz_path, allow_pickle=True)
clean_windows = data['clean']
subject_id = data['subject_id']

print(f"\nSubject ID: {subject_id}")
print(f"Shape of clean windows: {clean_windows.shape}")

# Select one window to visualize
clean_sample = clean_windows[0].copy()
win_len = clean_sample.shape[1]
time = np.arange(win_len) / 10.0 # sfreq is 10 Hz

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(time, clean_sample[0], color='red', label='HbO')
ax1.set_title(f'Clean HbO Signal Window (Subject: {subject_id})')
ax1.set_ylabel('Concentration (a.u.)')
ax1.legend()

ax2.plot(time, clean_sample[1], color='blue', label='HbR')
ax2.set_title(f'Clean HbR Signal Window')
ax2.set_ylabel('Concentration (a.u.)')
ax2.set_xlabel('Time (s)')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Explore Synthetic Noise Components

The `PLAN.md` specifies five types of synthetic noise to be added during training. Let's generate and visualize each one to understand their characteristics.

In [ ]:
n_samples = 128
sfreq = 10.0
rng = np.random.default_rng(42)

noise_funcs = {
    "Mayer Wave (~0.1 Hz)": synthesize_mayer_wave,
    "Respiratory (0.1-0.3 Hz)": synthesize_respiratory,
    "Cardiac (0.8-2.0 Hz)": synthesize_cardiac,
    "Motion Spikes": synthesize_motion_spikes,
    "Electronic (>2 Hz)": synthesize_electronic,
}

fig, axes = plt.subplots(len(noise_funcs), 1, figsize=(12, 15), sharex=True)
time = np.arange(n_samples) / sfreq

for i, (name, func) in enumerate(noise_funcs.items()):
    noise = func(n_samples, sfreq, rng)
    axes[i].plot(time, noise)
    axes[i].set_title(name)
    axes[i].set_ylabel('Amplitude (a.u.)')

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Examples of Synthetic Noise Components', fontsize=18, y=1.0)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

## 4. On-the-Fly Noise Injection

Now, let's see the `inject_noise` function in action. We'll take our clean sample, normalize it (as the `Dataset` class does), and then add a combination of the synthetic noises at a random Signal-to-Noise Ratio (SNR).

In [ ]:
# Take the clean sample from before
clean_norm = clean_sample.copy()

# Normalize it per-channel (as done in the Dataset class)
for c in range(clean_norm.shape[0]):
    var = clean_norm[c].var()
    if var >= 1e-12:
        clean_norm[c] = (clean_norm[c] - clean_norm[c].mean()) / np.sqrt(var)

# Inject noise at a specific SNR
snr_db = 5.0
noisy_sample, noise_info = inject_noise(clean_norm, snr_db, rng=np.random.default_rng(123))
noise_added = noisy_sample - clean_norm

print(f"Noise injected with SNR = {snr_db} dB")
print("Active noise types:", [k for k, v in noise_info.items() if v is True])

# Plot results for the HbO channel
ch_idx = 0
plt.figure(figsize=(14, 7))
plt.plot(time, clean_norm[ch_idx], label='Clean (Normalized)', color='green', linewidth=2, linestyle='--')
plt.plot(time, noisy_sample[ch_idx], label='Noisy', color='red', alpha=0.8)
plt.plot(time, noise_added[ch_idx], label='Added Noise', color='black', alpha=0.5)
plt.title(f'Clean vs. Noisy Signal (HbO, SNR={snr_db} dB)')
plt.xlabel('Time (s)')
plt.ylabel('Normalized Amplitude')
plt.legend()
plt.show()

## 5. Using the PyTorch `Dataset`

Finally, let's verify the full data loading pipeline by using the `FNIRSDenoiseDataset` class. This class handles loading the clean data, normalizing it, and injecting noise on-the-fly for each item requested by the `DataLoader`.

In [ ]:
# Instantiate the dataset for the subject we loaded earlier
dataset = FNIRSDenoiseDataset(
    npz_dir=PROCESSED_TRAIN_DIR,
    subject_ids=[str(subject_id)],
    snr_range=(-5.0, 20.0),
    seed=42
)

print(f"Dataset created for subject {subject_id} with {len(dataset)} windows.")

# Get a sample from the dataset
sample = dataset[0]
noisy_tensor = sample['noisy']
clean_tensor = sample['clean']
snr_db = sample['snr_db']

print(f"\nSample from dataset (SNR={snr_db:.2f} dB):")
print(f"Noisy tensor shape: {noisy_tensor.shape}")
print(f"Clean tensor shape: {clean_tensor.shape}")

# Plot the sample
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(time, clean_tensor[0].numpy(), 'g--', label='Clean Target')
ax1.plot(time, noisy_tensor[0].numpy(), 'r-', alpha=0.8, label='Noisy Input')
ax1.set_title(f'Dataset Sample - HbO (SNR={snr_db:.2f} dB)')
ax1.set_ylabel('Normalized Amplitude')
ax1.legend()

ax2.plot(time, clean_tensor[1].numpy(), 'g--', label='Clean Target')
ax2.plot(time, noisy_tensor[1].numpy(), 'b-', alpha=0.8, label='Noisy Input')
ax2.set_title(f'Dataset Sample - HbR')
ax2.set_ylabel('Normalized Amplitude')
ax2.set_xlabel('Time (s)')
ax2.legend()

plt.tight_layout()
plt.show()

## Conclusion

This notebook confirms that the training data pipeline is working as expected. We have:
1. Loaded preprocessed clean fNIRS windows.
2. Visualized the individual synthetic noise components.
3. Demonstrated the on-the-fly noise injection process.
4. Verified that the `FNIRSDenoiseDataset` correctly produces paired `(noisy, clean)` tensors for training.

The data generation pipeline is ready for use in Phase 4 (Training).